In [2]:
from transformers import MarianMTModel, MarianTokenizer

model_name = 'Helsinki-NLP/opus-mt-mul-en'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Define the text to be translated
# text = "Bonjour, comment allez-vous?"
text = "Programada muerte celular (PCD) es la muerte regulada de células dentro de un organismo. La planta de encaje (Aponogeton madagascariensis) produce perforaciones en sus hojas a través de PCD. Las hojas de la planta consisten en una red de venas longitudinales y transversales que encierran areolas. PCD ocurre en las células en el centro de estas areolas y progresa hacia afuera, deteniéndose aproximadamente a cinco células del sistema vascular."
translated = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt")
translated_tokens = model.generate(**translated)
translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

print(translated_text)


Scheduled cell death (PCD) is the regulated cell death within an organism. The cave plant (Aponogeton damagascarensis) produces perforations in its leaves through PCD. Plant leaves consist of a network of long-term and cross-sectional veins that will fill areas. PCD occurs in cells in the center of these areas and progress outside, holding approximately five cells of the vascular system.


In [16]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

# Load the tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
model = BertForMaskedLM.from_pretrained('bert-base-multilingual-cased')

# Tokenize the input text with a mask
text = "I want to [MASK] man"
inputs = tokenizer(text, return_tensors='pt')

# Forward pass through the model
outputs = model(**inputs)

# Extract the logits
logits = outputs.logits

# Get the logits for the masked token
mask_token_index = torch.where(inputs.input_ids == tokenizer.mask_token_id)[1]
mask_token_logits = logits[0, mask_token_index, :]

# Print the shape of the logits for the masked token
print("Logits shape for the masked token:", mask_token_logits.shape)



Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Logits shape for the masked token: torch.Size([1, 119547])


In [24]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# Load the mBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
model = AutoModelForMaskedLM.from_pretrained("bert-base-multilingual-cased")

# Define the document and query
doc = "Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature"
query = "What city is the Capital of France?"

# Define the function to get the max logits
def get_max_logits(output, tokens):
    return torch.max(
        torch.log(
            1 + torch.relu(output.logits)
        ) * tokens.attention_mask.unsqueeze(-1),
        dim=1)[0].squeeze().detach().cpu().numpy()

# Tokenize and get outputs for the document
tokens_doc = tokenizer(doc, return_tensors="pt")
output_doc = model(**tokens_doc)

# Tokenize and get outputs for the query
tokens_query = tokenizer(query, return_tensors="pt")
output_query = model(**tokens_query)

# Get the sparse representations
doc_vec = get_max_logits(output_doc, tokens_doc)
query_vec = get_max_logits(output_query, tokens_query)

print("Document Vector:", doc_vec)
print("Query Vector:", query_vec)

from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity([doc_vec, query_vec])


Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Document Vector: [0. 0. 0. ... 0. 0. 0.]
Query Vector: [0. 0. 0. ... 0. 0. 0.]


array([[1.0000013 , 0.52761453],
       [0.52761453, 1.0000002 ]], dtype=float32)

In [22]:
from transformers import BertTokenizer, BertForMaskedLM, AutoTokenizer, AutoModelForMaskedLM
import torch

# Load the mBERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
model = BertForMaskedLM.from_pretrained("bert-base-multilingual-cased")

# # Load the tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
# model = AutoModelForMaskedLM.from_pretrained("naver/splade-cocondenser-ensembledistil")


# Define the document and query in Spanish
doc = "Programada muerte celular (PCD) es la muerte regulada de células dentro de un organismo. La planta de encaje (Aponogeton madagascariensis) produce perforaciones en sus hojas a través de PCD. Las hojas de la planta consisten en una red de venas longitudinales y transversales que encierran areolas. PCD ocurre en las células en el centro de estas areolas y progresa hacia afuera, deteniéndose aproximadamente a cinco células del sistema vascular."
query = "What mechanism does the lace plant (Aponogeton madagascariensis) use to produce perforations in its leaves, and how does this process progress?"
query2 = "Combien de titres de La Liga a remporté le FC Barcelone?"
query3 = "What is Programmed Cell Death?"

# Define the function to get the max logits
def get_max_logits(output, tokens):
    return torch.max(
        torch.log(
            1 + torch.relu(output.logits)
        ) * tokens.attention_mask.unsqueeze(-1),
        dim=1)[0].squeeze().detach().cpu().numpy()

# Tokenize and get outputs for the document
tokens_doc = tokenizer(doc, return_tensors="pt")
output_doc = model(**tokens_doc)

# Tokenize and get outputs for the query
tokens_query = tokenizer(query, return_tensors="pt")
output_query = model(**tokens_query)

tokens_query2 = tokenizer(query2, return_tensors="pt")
output_query2 = model(**tokens_query2)

tokens_query3 = tokenizer(query3, return_tensors="pt")
output_query3 = model(**tokens_query3)

# Get the sparse representations
doc_vec = get_max_logits(output_doc, tokens_doc)
query_vec = get_max_logits(output_query, tokens_query)
query_vec2 = get_max_logits(output_query2, tokens_query2)
query_vec3 = get_max_logits(output_query3, tokens_query3)


Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [23]:
from sklearn.metrics.pairwise import cosine_similarity

print(cosine_similarity([doc_vec, query_vec]))
print(cosine_similarity([doc_vec, query_vec, query_vec2]))
print(cosine_similarity([doc_vec, query_vec, query_vec2, query_vec3]))

[[1.0000029 0.7353121]
 [0.7353121 1.       ]]
[[1.0000029 0.7353121 0.5496435]
 [0.7353121 1.        0.5540756]
 [0.5496435 0.5540756 1.0000001]]
[[1.0000029  0.7353121  0.5496435  0.4463375 ]
 [0.7353121  1.         0.5540756  0.542318  ]
 [0.5496435  0.5540756  1.0000001  0.48366252]
 [0.4463375  0.542318   0.48366252 0.99999994]]


In [6]:
# Define the function to get the max logits
def get_max_logits(output, tokens):
    return torch.max(
        torch.log(
            1 + torch.relu(output.logits)
        ) * tokens.attention_mask.unsqueeze(-1),
        dim=1)[0].squeeze().detach().cpu().numpy()

def get_sparse_embeddings(tokenizer, model, text):
    tokens = tokenizer(text, return_tensors="pt")
    output = model(**tokens)
    return get_max_logits(output, tokens)

In [26]:
from transformers import BertTokenizer, BertForMaskedLM, AutoTokenizer, AutoModelForMaskedLM
import torch
from scipy.spatial import distance
from sklearn.metrics.pairwise import cosine_similarity

# Load the mBERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
model = BertForMaskedLM.from_pretrained("bert-base-multilingual-cased")

tokenizer2 = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
model2 = AutoModelForMaskedLM.from_pretrained("naver/splade-cocondenser-ensembledistil")

tokenizer3 = AutoTokenizer.from_pretrained("bert-base-uncased")
model3 = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")

doc_eng = "Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature"
doc_es = "Programada muerte celular (PCD) es la muerte regulada de células dentro de un organismo. La planta de encaje (Aponogeton madagascariensis) produce perforaciones en sus hojas a través de PCD. Las hojas de la planta consisten en una red de venas longitudinales y transversales que encierran areolas. PCD ocurre en las células en el centro de estas areolas y progresa hacia afuera, deteniéndose aproximadamente a cinco células del sistema vascular."
query1 = "What mechanism does the lace plant (Aponogeton madagascariensis) use to produce perforations in its leaves, and how does this process progress?"
query2 = "Cual es la capital de Francia?"
query3 = "What is Programmed Cell Death?"

Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint o

In [59]:
output = tokenizer(doc_es, return_tensors="pt")
output2 = tokenizer2(doc_es, return_tensors="pt")
output3 = tokenizer3(doc_es, return_tensors="pt")

result1 = model(**output)
result2 = model2(**output2)
result3 = model3(**output3)

doc_vec = get_max_logits(result1, output)
doc_vec2 = get_max_logits(result2, output2)
doc_vec3 = get_max_logits(result3, output3)

In [60]:
print(result1.logits.shape, result2.logits.shape, result3.logits.shape)
print("Nonzeros in the vectors")
print("doc_vec:", len(doc_vec[doc_vec != 0]), len(doc_vec))
print("doc_vec:", len(doc_vec2[doc_vec2 != 0]), len(doc_vec2))
print("doc_vec:", len(doc_vec3[doc_vec3 != 0]), len(doc_vec3))

torch.Size([1, 113, 119547]) torch.Size([1, 144, 30522]) torch.Size([1, 144, 30522])
Nonzeros in the vectors
doc_vec: 86763 119547
doc_vec: 153 30522
doc_vec: 24506 30522


In [18]:
doc_vec_eng = get_sparse_embeddings(tokenizer, model, doc_eng)
doc_vec_es = get_sparse_embeddings(tokenizer, model, doc_es)
query_vec1 = get_sparse_embeddings(tokenizer, model, query1)
query_vec2 = get_sparse_embeddings(tokenizer, model, query2)
query_vec3 = get_sparse_embeddings(tokenizer, model, query3)

In [19]:
doc_vec_eng2 = get_sparse_embeddings(tokenizer2, model2, doc_eng)
doc_vec_es2 = get_sparse_embeddings(tokenizer2, model2, doc_es)
query_vec1_2 = get_sparse_embeddings(tokenizer2, model2, query1)
query_vec2_2 = get_sparse_embeddings(tokenizer2, model2, query2)
query_vec3_2 = get_sparse_embeddings(tokenizer2, model2, query3)

In [36]:
doc_vec_eng3 = get_sparse_embeddings(tokenizer3, model3, doc_eng)
doc_vec_es3 = get_sparse_embeddings(tokenizer3, model3, doc_es)
query_vec1_3 = get_sparse_embeddings(tokenizer3, model3, query1)
query_vec2_3 = get_sparse_embeddings(tokenizer3, model3, query2)
query_vec3_3 = get_sparse_embeddings(tokenizer3, model3, query3)

In [54]:
# calculate nonzeros in the vectors
print("Nonzeros in the vectors")
print("doc_vec_es:", len(doc_vec_es[doc_vec_es != 0]), len(doc_vec_es))
print("doc_vec_es:", len(doc_vec_es2[doc_vec_es2 != 0]), len(doc_vec_es2))
print("doc_vec_es:", len(doc_vec_es3[doc_vec_es3 != 0]), len(doc_vec_es3))




Nonzeros in the vectors
doc_vec_es: 86763 119547
doc_vec_es: 153 30522
doc_vec_es: 24506 30522


In [21]:
print(cosine_similarity([doc_vec_eng, doc_vec_es]))
# print(distance.cosine(doc_vec_eng, doc_vec_es))
# print(cosine_similarity([doc_vec_eng, doc_vec_es, query_vec1, query_vec2, query_vec3]))

[[1.0000013  0.85895747]
 [0.85895747 1.0000029 ]]
0.14104423883418926
[[1.0000013  0.85895747 0.8424771  0.52034795 0.54407567]
 [0.85895747 1.0000029  0.7353121  0.54811025 0.4463375 ]
 [0.8424771  0.7353121  1.         0.52228117 0.542318  ]
 [0.52034795 0.54811025 0.52228117 1.0000006  0.42810035]
 [0.54407567 0.4463375  0.542318   0.42810035 0.99999994]]


In [22]:
print(cosine_similarity([doc_vec_eng2, doc_vec_es2]))
print(distance.cosine(doc_vec_eng2, doc_vec_es2))
print(cosine_similarity([doc_vec_eng2, doc_vec_es2, query_vec1_2, query_vec2_2, query_vec3_2], ))

[[0.99999994 0.38303924]
 [0.38303924 0.99999994]]
0.6169607414172045
[[9.9999994e-01 3.8303924e-01 5.7648879e-01 4.6799413e-04 4.5484224e-01]
 [3.8303924e-01 9.9999994e-01 3.1879342e-01 7.0458256e-02 3.6124546e-02]
 [5.7648879e-01 3.1879342e-01 1.0000000e+00 0.0000000e+00 3.6697380e-02]
 [4.6799413e-04 7.0458256e-02 0.0000000e+00 1.0000000e+00 9.7476062e-04]
 [4.5484224e-01 3.6124546e-02 3.6697380e-02 9.7476062e-04 1.0000000e+00]]


In [23]:
print(doc_eng)
print(doc_es)
print(query1)
print(query2)
print(query3)

Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature
Programada muerte celular (PCD) es la muerte regulada de células dentro de un organismo. La planta de encaje (Aponogeton madagascariensis) produce perforaciones en sus hojas a través de PCD. Las hojas de la planta consisten en una red de venas longitudinales y transversales que encierran areolas. PCD ocurre en las células en el centro de estas areolas y progresa hacia afuera, deteniéndose aproximadamente a cinco células del sistema vascular.
What mechanism does the lace plant (Aponogeton madagascariensis) use to produce perforations in its leaves, and how does this process progre

In [34]:
import numpy as np

sim = np.zeros((doc_vec_eng.shape[0], doc_vec_eng.shape[0]))

for i, vec in enumerate(doc_vec_eng):
    sim[i,:] = np.dot(doc_vec_es, doc_vec_eng.T) / (
        np.linalg.norm(doc_vec_es) * np.linalg.norm(doc_vec_eng.reshape(1, -1))
    )
sim

sim = np.zeros((doc_vec_eng2.shape[0], doc_vec_eng2.shape[0]))

for i, vec in enumerate(doc_vec_eng2):
    sim[i,:] = np.dot(doc_vec_es2, doc_vec_eng2.T) / (
        np.linalg.norm(doc_vec_es2) * np.linalg.norm(doc_vec_eng2.reshape(1, -1))
    )
sim
